# Gaussian external constraints

Constraints now attach directly to a `FitSession`; no manual `ConstrainedNLL` wrapping is needed in the analysis code.


In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    BackgroundSpec, DecayChannel, DecayModel, FitSession, GaussianConstraint,
    NonResonant, Parameter, RealImag, ToyBackground, enable_x64, generate_toy,
)
from dalitzplotfitter.background import FunctionalBackground

enable_x64()


In [ ]:
model=DecayModel(
    DecayChannel("B+",("K+","pi+","pi-")),
    [NonResonant(RealImag(1.0,0.0))],
    normalization_method="square-dalitz",normalization_resolution=150,normalization_pair=(0,2),
)
background=FunctionalBackground(lambda d:0.4+0.6*jnp.clip(d["s23"]/25,0,1))
data=generate_toy(
    model,20_000,signal_fraction=0.68,
    backgrounds=(ToyBackground("comb",background),),
    seed=1111,
)
f_sig=Parameter("signal_fraction",0.55,bounds=(0.05,0.98),step=0.01)
base=FitSession(
    model,data,signal_fraction=f_sig,
    backgrounds=(BackgroundSpec("comb",background),),
)
constrained=base.with_constraint(GaussianConstraint(f_sig,mean=0.70,sigma=0.04))


In [ ]:
free_result=base.fit()
constrained_result=constrained.fit()

print("free")
base.report(free_result,include_fit_fractions=False)
print("\nconstrained")
constrained.report(constrained_result,include_fit_fractions=False)

scan=np.linspace(0.50,0.85,120)
free=np.asarray([base.objective({"signal_fraction":x}) for x in scan])
cons=np.asarray([constrained.objective({"signal_fraction":x}) for x in scan])
plt.plot(scan,free-free.min(),label="free")
plt.plot(scan,cons-cons.min(),label="constrained")
plt.axvline(0.68,ls="--",label="truth")
plt.xlabel("$f_{sig}$")
plt.ylabel(r"$\Delta$NLL")
plt.legend()
plt.show()
